# Just-in-time Qwen neuron lens

This notebook calibrates every MLP neuron once, ranks normalized positive activations for a sentence, and profiles a chosen neuron on OpenWebText only when requested.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "jit_neuron_lens.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from jit_neuron_lens import JITNeuronLens

lens = JITNeuronLens(cache_dir=PROJECT_ROOT / "outputs/jit_neuron_lens")

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

## One-time normalization calibration

We estimate a mean and standard deviation for every `(layer, neuron)` over an OpenWebText sample. This makes scores comparable across neurons and layers. The result is cached and automatically reused.

In [2]:
if lens.mean is None or lens.maximum is None:
    lens.calibrate(num_samples=10_000, max_length=256, batch_size=16)

## One-time top-occurrence index

This scans the corpus once and stores only the top 20 absolute-z locations per neuron plus token IDs. Use `layers=7`, `layers=[7, 16]`, or `layers="all"`. A selected subset stops the forward pass after its highest layer. Later neuron profiles rerun only their selected contexts.

In [3]:
index_path = lens.build_occurrence_index(
    num_samples=10_000,
    candidates_per_neuron=20,
    max_length=256,
    batch_size=32,
    layers="all",
)
index_path

using existing index: /home/liusida/research/goodfire/already-clean/outputs/jit_neuron_lens/occurrence-index-v2-tails-all-n10000-m256-s42-k20.pt


PosixPath('outputs/jit_neuron_lens/occurrence-index-v2-tails-all-n10000-m256-s42-k20.pt')

## Analyze a sentence

Choose a layer explicitly. Each token shows its three largest absolute standardized neuron activations within that layer. Click any neuron badge to load its indexed positive- and negative-tail occurrences below the token cards.

In [8]:
analysis = lens.analyze(
    "Today is Monday, so tomorrow will be Tuesday.\nNot today.",
    layer=1,
    top_k=3,
)
analysis

## Profile a chosen neuron

Set the layer and neuron from the analysis above, then execute the cell. Results are cached by neuron and settings. Context tokens use a white-to-yellow absolute-z heatmap.

In [5]:
profile = lens.profile_neuron(
    layer=3,
    neuron=2595,
    num_samples=10_000,
    top_k=10,
    max_length=256,
    batch_size=32,
    context_tokens=24,
)
profile

NeuronProfile(layer=3, neuron=2595, samples_processed=10000, occurrences=[{'tail': 'positive', 'activation': 1.0390625, 'z_score': 21.730512619018555, 'sample_index': 5661, 'token_position': 98, 'active_positions': [98], 'occurrence_count': 1, 'context_tokens': [{'text': ' same', 'token_id': 1788, 'token_position': 74, 'activation': -0.05029296875, 'z_score': -0.987074613571167, 'is_active': False}, {'text': ' thing', 'token_id': 3065, 'token_position': 75, 'activation': -0.00543212890625, 'z_score': -0.05153968185186386, 'is_active': False}, {'text': ' they', 'token_id': 781, 'token_position': 76, 'activation': 0.03759765625, 'z_score': 0.8458101153373718, 'is_active': False}, {'text': ' do', 'token_id': 635, 'token_position': 77, 'activation': 0.154296875, 'z_score': 3.2794737815856934, 'is_active': False}, {'text': ' at', 'token_id': 506, 'token_position': 78, 'activation': 0.006072998046875, 'z_score': 0.18839001655578613, 'is_active': False}, {'text': ' work', 'token_id': 944, 'token_position': 79, 'activation': 0.031494140625, 'z_score': 0.7185264229774475, 'is_active': False}, {'text': ',', 'token_id': 11, 'token_position': 80, 'activation': 0.0230712890625, 'z_score': 0.5428750514984131, 'is_active': False}, {'text': ' just', 'token_id': 1066, 'token_position': 81, 'activation': -0.0030517578125, 'z_score': -0.0018990535754710436, 'is_active': False}, {'text': ' at', 'token_id': 506, 'token_position': 82, 'activation': -0.003875732421875, 'z_score': -0.01908234693109989, 'is_active': False}, {'text': ' home', 'token_id': 2046, 'token_position': 83, 'activation': 5.7697296142578125e-05, 'z_score': 0.06294600665569305, 'is_active': False}, {'text': '.', 'token_id': 13, 'token_position': 84, 'activation': 0.0169677734375, 'z_score': 0.41559135913848877, 'is_active': False}, {'text': ' For', 'token_id': 1690, 'token_position': 85, 'activation': 0.025390625, 'z_score': 0.5912428498268127, 'is_active': False}, {'text': ' example', 'token_id': 3010, 'token_position': 86, 'activation': 0.353515625, 'z_score': 7.434012413024902, 'is_active': False}, {'text': ',', 'token_id': 11, 'token_position': 87, 'activation': 0.107421875, 'z_score': 2.3019351959228516, 'is_active': False}, {'text': ' I', 'token_id': 353, 'token_position': 88, 'activation': 0.1142578125, 'z_score': 2.444493055343628, 'is_active': False}, {'text': ' work', 'token_id': 944, 'token_position': 89, 'activation': 0.3671875, 'z_score': 7.719127655029297, 'is_active': False}, {'text': ' in', 'token_id': 303, 'token_position': 90, 'activation': 0.06494140625, 'z_score': 1.4160410165786743, 'is_active': False}, {'text': ' a', 'token_id': 264, 'token_position': 91, 'activation': 0.05126953125, 'z_score': 1.1309255361557007, 'is_active': False}, {'text': ' department', 'token_id': 9016, 'token_position': 92, 'activation': 0.051025390625, 'z_score': 1.1258341073989868, 'is_active': False}, {'text': ' filled', 'token_id': 9896, 'token_position': 93, 'activation': -0.003326416015625, 'z_score': -0.007626818493008614, 'is_active': False}, {'text': ' with', 'token_id': 440, 'token_position': 94, 'activation': -0.012939453125, 'z_score': -0.20809859037399292, 'is_active': False}, {'text': ' fish', 'token_id': 7397, 'token_position': 95, 'activation': 0.00286865234375, 'z_score': 0.12156609445810318, 'is_active': False}, {'text': ' bi', 'token_id': 5843, 'token_position': 96, 'activation': -2.944469451904297e-05, 'z_score': 0.061128731817007065, 'is_active': False}, {'text': 'ologists', 'token_id': 20651, 'token_position': 97, 'activation': 0.053955078125, 'z_score': 1.1869302988052368, 'is_active': False}, {'text': '.', 'token_id': 13, 'token_position': 98, 'activation': 1.0390625, 'z_score': 21.730512619018555, 'is_active': True}, {'text': ' Those', 'token_id': 12597, 'token_position': 99, 'activation': 0.049072265625, 'z_score': 1.0851033926010132, 'is_active': False}, {'text': ' fish', 'token_id': 7397, 'token_position': 100, 'activation': -0.10546875, 'z_score': -2.13771

### Faster exploratory run

While iterating on the workflow, use `num_samples=10_000`; increase it to 100,000 for the final profile.